# 🐉 Descarga + Estandarización — Dragon Ball (clase «DB»)
**ExpoEscom — Clasificador Multietiqueta** · v1.0.0

Descarga varios datasets de Dragon Ball **directamente al I/O de Colab**
(`/content/`, efímero), los estandariza a `224×224` RGB con la misma lógica
que `01_estandarizar`, y guarda el resultado en
`/content/drive/MyDrive/ExpoEscom/dataset/DB/` como `DB_NNNNNN.jpg`.

**Meta:** 50,000 imágenes (reales primero; augmentadas solo para rellenar).

| Fuente | Origen | Credencial |
|---|---|---|
| Carpeta de Drive (**PRINCIPAL**) | Google Drive | — (auth Colab) |
| dragon-ball-super-saiyan-dataset | Kaggle | kaggle.json |
| dragon-ball-anime-dataset | Kaggle | kaggle.json |
| dragon-ball-z-dataset | Kaggle | kaggle.json |
| krigeta/dragonballsuper | HuggingFace | — (público) |
| dbs-o7llq | Roboflow | API key |
| emotion-dbz | Roboflow | API key |

**Orden:** CELDA 1 (setup) → 2 (config) → 3 (credenciales) → 4 (descarga)
→ 5 (funciones) → 6 (estandarizar) → 7 (verificación) → 8 (limpieza).

## CELDA 1 — Dependencias, versión y montaje de Drive

In [ ]:
VERSION = '1.0.0'

print('═' * 50)
print('🐉 Descarga + Estandarización — Dragon Ball (DB)')
print(f'v{VERSION}')
print('═' * 50)

!apt-get install -y unrar > /dev/null 2>&1
!pip install -q gdown kaggle roboflow huggingface_hub opencv-python-headless PyDrive2

import os, io, json, shutil, zipfile, subprocess, random, time, threading, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image, ImageEnhance
import cv2
from google.colab import drive, auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore', category=UserWarning, module='PIL')

drive.mount('/content/drive', force_remount=False)
auth.authenticate_user()
service = build('drive', 'v3')
print('✅ Dependencias + Drive + API listos')

## CELDA 2 — Configuración (solo modifica aquí)

In [ ]:
# ════ CONFIGURACIÓN ═══════════════════════════════════════════
CLASE           = 'DB'
MAX_IMAGENES    = 50_000
DATASET_DESTINO = '/content/drive/MyDrive/ExpoEscom/dataset'   # mismo destino que 01

# ── Fuente PRINCIPAL: carpeta de Google Drive (compartida por enlace) ──
DRIVE_FOLDER_ID = '1fNiC8WVKgsy5SzNjGXQANnoEx-sFIsgS'

# ── Datasets de Kaggle (slug) — necesita kaggle.json (CELDA 3) ──
KAGGLE_DATASETS = [
    'bhav09/dragon-ball-super-saiyan-dataset',
    'amlanmohanty1/dragon-ball-anime-dataset',
    'insaiyancvk/dragon-ball-z-dataset',
]

# ── Datasets de HuggingFace (repo_id) — públicos, sin token ──
HF_DATASETS = [
    'krigeta/dragonballsuper',
]

# ── Datasets de Roboflow (workspace, project, version) — necesita API key (CELDA 3) ──
ROBOFLOW_DATASETS = [
    ('uao-wjax1',       'dbs-o7llq',  1),
    ('converter-sq0eo', 'emotion-dbz', 1),
]

# ── Paralelismo (Colab Pro+: ~12 vCPU) ──
MAX_WORKERS      = 12
DOWNLOAD_WORKERS = 4

# ── Augmentación (relleno SOLO si las fuentes reales no llegan a la meta) ──
GARANTIZAR_EXACTO = True
PERMITIR_AUGMENT  = True

# ── Rutas (I/O de Colab — efímero) ──
IMG_SIZE        = (224, 224)
SEED            = 42
RAW_DIR         = Path('/content/db_raw')          # staging de descargas
DESTINO_CLASE   = Path(DATASET_DESTINO) / CLASE    # → /dataset/DB
CHECKPOINT_PATH = DESTINO_CLASE / '_checkpoint_DB.json'

# MP4 y GIF: se extrae un frame y se guarda como JPG
EXTENSIONES_IMAGEN = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif', '.mp4'}

RAW_DIR.mkdir(parents=True, exist_ok=True)
DESTINO_CLASE.mkdir(parents=True, exist_ok=True)
random.seed(SEED)

print(f'Clase   : {CLASE}')
print(f'Meta    : {MAX_IMAGENES:,}')
print(f'Destino : {DESTINO_CLASE}')
print(f'Fuentes : 1 Drive (principal) · {len(KAGGLE_DATASETS)} Kaggle · '
      f'{len(HF_DATASETS)} HF · {len(ROBOFLOW_DATASETS)} Roboflow')

## CELDA 3 — Credenciales (Kaggle y Roboflow)

- **Kaggle:** pega tu usuario/clave abajo, o deja vacío y sube `kaggle.json`
  (Kaggle → *Account* → *Create New API Token*).
- **Roboflow:** pega tu API key (Roboflow → *Settings* → *API*). Si la dejas
  vacía, las fuentes de Roboflow se saltan (aportan pocas imágenes).

In [ ]:
# ── Kaggle ──
KAGGLE_USERNAME = ''   # ← opcional (si lo dejas vacío, se pedirá subir kaggle.json)
KAGGLE_KEY      = ''   # ← opcional

if KAGGLE_USERNAME and KAGGLE_KEY:
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
    os.environ['KAGGLE_KEY']      = KAGGLE_KEY
    print('✅ Kaggle: credenciales desde variables')
elif Path('/root/.kaggle/kaggle.json').exists() or Path('/content/kaggle.json').exists():
    _src = '/content/kaggle.json' if Path('/content/kaggle.json').exists() else '/root/.kaggle/kaggle.json'
    Path('/root/.kaggle').mkdir(parents=True, exist_ok=True)
    shutil.copy(_src, '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('✅ Kaggle: kaggle.json detectado')
else:
    print('⚠️ Kaggle sin credenciales — sube tu kaggle.json:')
    try:
        from google.colab import files
        _up = files.upload()                       # selecciona kaggle.json
        Path('/root/.kaggle').mkdir(parents=True, exist_ok=True)
        shutil.move(next(iter(_up)), '/root/.kaggle/kaggle.json')
        os.chmod('/root/.kaggle/kaggle.json', 0o600)
        print('✅ Kaggle: kaggle.json subido')
    except Exception as e:
        print(f'   (omitido: {e}) — las fuentes de Kaggle se saltarán')

# ── Roboflow ──
ROBOFLOW_API_KEY = ''   # ← pega tu API key (o deja vacío para saltar Roboflow)
print('✅ Roboflow:', 'API key lista' if ROBOFLOW_API_KEY else 'sin key (se salta)')

## CELDA 4 — Descargar TODAS las fuentes al I/O de Colab (`/content/db_raw`)

In [ ]:
# ════ DESCARGA → /content/db_raw  (cada fuente tolera fallos por separado) ════

def _extraer_archivo(path, destino):
    # Extrae .zip / .rar dentro de `destino`. Devuelve True si extrajo algo.
    ext = Path(path).suffix.lower()
    try:
        destino.mkdir(parents=True, exist_ok=True)
        if ext == '.zip':
            with zipfile.ZipFile(path) as zf:
                zf.extractall(destino)
            return True
        if ext == '.rar':
            subprocess.run(['unrar', 'x', '-o+', str(path), str(destino) + '/'],
                           capture_output=True, timeout=7200)
            return True
    except Exception as e:
        print(f'   ⚠️ No se pudo extraer {Path(path).name}: {str(e)[:120]}')
    return False


# ── 1) Fuente PRINCIPAL: carpeta de Drive (vía API, recursiva) ──
_FOLDER_MIME = 'application/vnd.google-apps.folder'
_ARGS = dict(supportsAllDrives=True, includeItemsFromAllDrives=True)

def _listar_drive(folder_id):
    archivos, stack = [], [folder_id]
    while stack:
        fid, token = stack.pop(), None
        while True:
            resp = service.files().list(
                q=f"'{fid}' in parents and trashed=false",
                fields='nextPageToken, files(id,name,mimeType,size)',
                pageToken=token, **_ARGS).execute()
            for f in resp.get('files', []):
                if f['mimeType'] == _FOLDER_MIME:
                    stack.append(f['id'])
                else:
                    archivos.append(f)
            token = resp.get('nextPageToken')
            if not token:
                break
    return archivos

def _descargar_drive_file(f, destino_dir):
    dest = destino_dir / f['name']
    if dest.exists() and dest.stat().st_size > 0:
        return 'caché'
    try:
        req = service.files().get_media(fileId=f['id'], supportsAllDrives=True)
        with io.FileIO(dest, 'wb') as fh:
            dl = MediaIoBaseDownload(fh, req, chunksize=50 * 1024 * 1024)
            done = False
            while not done:
                _, done = dl.next_chunk()
        return 'ok'
    except Exception as e:
        if dest.exists():
            dest.unlink()
        return f'❌ {str(e)[:80]}'

def descargar_drive():
    out = RAW_DIR / 'drive_principal'
    out.mkdir(parents=True, exist_ok=True)
    print('⬇️  Drive principal: listando…')
    try:
        archivos = _listar_drive(DRIVE_FOLDER_ID)
    except Exception as e:
        print(f'   ❌ No se pudo listar la carpeta de Drive: {str(e)[:160]}')
        return
    print(f'   {len(archivos):,} archivos en la carpeta')
    libre_gb = shutil.disk_usage('/content').free / (1024 ** 3)
    print(f'   Espacio libre en /content: {libre_gb:.1f} GB')
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as ex:
        list(tqdm(ex.map(lambda f: _descargar_drive_file(f, out), archivos),
                  total=len(archivos), desc='Drive', unit='archivo'))
    for arch in list(out.glob('*.zip')) + list(out.glob('*.rar')):
        print(f'   📦 extrayendo {arch.name}…')
        _extraer_archivo(arch, out / (arch.stem + '_x'))
        arch.unlink(missing_ok=True)   # liberar espacio tras extraer
    print('   ✅ Drive principal listo')


# ── 2) Kaggle ──
def descargar_kaggle():
    if not (Path('/root/.kaggle/kaggle.json').exists() or os.environ.get('KAGGLE_KEY')):
        print('⚠️ Kaggle: sin credenciales, se salta.')
        return
    for slug in KAGGLE_DATASETS:
        dst = RAW_DIR / ('kaggle_' + slug.split('/')[-1])
        dst.mkdir(parents=True, exist_ok=True)
        print(f'⬇️  Kaggle: {slug}…')
        r = subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', slug, '-p', str(dst), '--unzip'],
            capture_output=True, text=True)
        if r.returncode != 0:
            print(f'   ⚠️ {slug}: {r.stderr.strip()[:200]}')
        else:
            print(f'   ✅ {slug}')


# ── 3) HuggingFace ──
def descargar_hf():
    from huggingface_hub import snapshot_download
    for repo in HF_DATASETS:
        dst = RAW_DIR / ('hf_' + repo.split('/')[-1])
        print(f'⬇️  HF: {repo}…')
        try:
            snapshot_download(repo_id=repo, repo_type='dataset',
                              local_dir=str(dst), local_dir_use_symlinks=False)
            print(f'   ✅ {repo}')
        except Exception as e:
            print(f'   ⚠️ {repo}: {str(e)[:200]}')


# ── 4) Roboflow ──
def descargar_roboflow():
    if not ROBOFLOW_API_KEY:
        print('⚠️ Roboflow: sin API key, se salta.')
        return
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    for ws, proj, ver in ROBOFLOW_DATASETS:
        print(f'⬇️  Roboflow: {ws}/{proj} v{ver}…')
        try:
            p = rf.workspace(ws).project(proj)
            p.version(ver).download('folder', location=str(RAW_DIR / ('rf_' + proj)))
            print(f'   ✅ {proj}')
        except Exception as e:
            print(f'   ⚠️ {proj}: {str(e)[:200]}')


descargar_drive()
descargar_kaggle()
descargar_hf()
descargar_roboflow()

_crudas = [p for p in RAW_DIR.rglob('*')
           if p.is_file() and p.suffix.lower() in EXTENSIONES_IMAGEN]
print(f'\n📦 Total archivos imagen/vídeo crudos recolectados: {len(_crudas):,}')

## CELDA 5 — Funciones de estandarización (misma lógica que 01)

In [ ]:
def es_imagen(nombre):
    return Path(nombre).suffix.lower() in EXTENSIONES_IMAGEN


def _abrir_como_imagen(ruta):
    # static → RGB ; GIF animado → frame aleatorio ; MP4 → frame del primer tercio
    ext = Path(ruta).suffix.lower()
    if ext == '.mp4':
        try:
            cap = cv2.VideoCapture(str(ruta))
            total = max(1, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
            cap.set(cv2.CAP_PROP_POS_FRAMES, random.randint(0, total // 3))
            ok, frame = cap.read()
            cap.release()
            if not ok:
                return None
            return Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        except Exception:
            return None
    try:
        img = Image.open(ruta)
        if ext == '.gif':
            n = getattr(img, 'n_frames', 1)
            if n > 1:
                try:
                    img.seek(random.randint(0, n - 1))
                except EOFError:
                    img.seek(0)
        return img.convert('RGB')
    except Exception:
        return None


# Índice global thread-safe para nombrar sin colisiones
_lock_idx = threading.Lock()
_idx_global = [0]
def _siguiente_indice():
    with _lock_idx:
        i = _idx_global[0]
        _idx_global[0] += 1
    return i

# Contador global thread-safe (corte EXACTO en MAX_IMAGENES)
_lock_total = threading.Lock()
_total_guardadas = [0]
def _reservar_cupo():
    with _lock_total:
        if _total_guardadas[0] >= MAX_IMAGENES:
            return False
        _total_guardadas[0] += 1
        return True
def _liberar_cupo():
    with _lock_total:
        _total_guardadas[0] -= 1
def _cupo_restante():
    with _lock_total:
        return max(0, MAX_IMAGENES - _total_guardadas[0])

def _inicializar_indice():
    # Resume: cuenta lo que ya existe en destino para continuar (no re-hacer)
    maxidx, n = -1, 0
    for f in DESTINO_CLASE.glob(f'{CLASE}_*.jpg'):
        n += 1
        try:
            maxidx = max(maxidx, int(f.stem[len(CLASE) + 1:]))
        except ValueError:
            pass
    _idx_global[0] = maxidx + 1
    _total_guardadas[0] = n
_inicializar_indice()


def procesar_una(ruta, pbar):
    # reserva cupo → abre/estandariza → guarda DB_NNNNNN.jpg
    if not _reservar_cupo():
        return False
    img = _abrir_como_imagen(ruta)
    if img is None:
        _liberar_cupo()
        return False
    try:
        idx = _siguiente_indice()
        dest = DESTINO_CLASE / f'{CLASE}_{idx:06d}.jpg'
        img.resize(IMG_SIZE, Image.LANCZOS).save(dest, 'JPEG', quality=90, optimize=True)
        pbar.update(1)
        return True
    except Exception:
        _liberar_cupo()
        return False


def cargar_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f:
                return json.load(f)
        except Exception:
            return {}
    return {}
def guardar_checkpoint(c):
    tmp = str(CHECKPOINT_PATH) + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(c, f, indent=2)
    os.replace(tmp, str(CHECKPOINT_PATH))


print(f'✅ Funciones listas | ya en destino: {_total_guardadas[0]:,} | '
      f'próximo índice: {_idx_global[0]:,}')

## CELDA 6 — Estandarizar a 50k (reales primero · augmentar para rellenar)

In [ ]:
ya = _total_guardadas[0]
if ya >= MAX_IMAGENES:
    print(f'✅ Ya hay {ya:,} imágenes en destino. Nada que hacer.')
else:
    print('🔎 Recolectando imágenes crudas de /content/db_raw…')
    rutas = [p for p in RAW_DIR.rglob('*') if p.is_file() and es_imagen(p.name)]
    random.shuffle(rutas)
    print(f'   {len(rutas):,} archivos imagen/vídeo encontrados')

    pbar = tqdm(total=MAX_IMAGENES, initial=ya, desc='Estandarizando', unit='img')

    # ── PASO 1: estandarización PARALELA con corte exacto en 50k ──
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        for _ in ex.map(lambda r: procesar_una(r, pbar), rutas):
            if _cupo_restante() <= 0:
                break

    n_aug = 0

    # ── PASO 2: augmentación de relleno (último recurso) ──
    if GARANTIZAR_EXACTO and PERMITIR_AUGMENT and _cupo_restante() > 0:
        objetivo = _cupo_restante()
        print(f'\n🎨 Fuentes agotadas — augmentando {objetivo:,} para completar {MAX_IMAGENES:,}')
        base = sorted(DESTINO_CLASE.glob(f'{CLASE}_*.jpg'))

        def augmentar(img):
            op = random.choice(['flip', 'rot', 'bright', 'contrast', 'zoom', 'combo'])
            if op in ('flip', 'combo'):
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
            if op in ('rot', 'combo'):
                img = img.rotate(random.uniform(-18, 18), resample=Image.BILINEAR, expand=False)
            if op in ('bright', 'combo'):
                img = ImageEnhance.Brightness(img).enhance(random.uniform(0.75, 1.25))
            if op in ('contrast', 'combo'):
                img = ImageEnhance.Contrast(img).enhance(random.uniform(0.75, 1.25))
            if op in ('zoom', 'combo'):
                w, h = img.size
                fz = random.uniform(0.8, 0.95)
                cw, ch = int(w * fz), int(h * fz)
                x, y = random.randint(0, w - cw), random.randint(0, h - ch)
                img = img.crop((x, y, x + cw, y + ch)).resize((w, h), Image.LANCZOS)
            return img

        def _aug_one(_):
            if not _reservar_cupo():
                return False
            try:
                with Image.open(random.choice(base)) as im:
                    a = augmentar(im.convert('RGB'))
                idx = _siguiente_indice()
                a.resize(IMG_SIZE, Image.LANCZOS).save(
                    DESTINO_CLASE / f'{CLASE}_{idx:06d}.jpg', 'JPEG', quality=90, optimize=True)
                return True
            except Exception:
                _liberar_cupo()
                return False

        if base:
            with tqdm(total=objetivo, desc='Augmentando', unit='img') as pb:
                while _cupo_restante() > 0:
                    antes = n_aug
                    faltan = _cupo_restante()
                    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
                        for ok in ex.map(_aug_one, range(faltan)):
                            if ok:
                                n_aug += 1
                                pbar.update(1)
                                pb.update(1)
                    if n_aug == antes:
                        print('   ⚠️ No se pudo augmentar más (imágenes base con errores).')
                        break
        else:
            print('   ⚠️ No hay imágenes base para augmentar (¿descargas vacías?).')

    pbar.close()

    total_final = len(list(DESTINO_CLASE.glob(f'{CLASE}_*.jpg')))
    print('\n' + '═' * 50)
    print('RESUMEN FINAL')
    print('═' * 50)
    print(f'  En disco      : {total_final:,}')
    print(f'    · reales    : {total_final - n_aug:,}')
    print(f'    · augment.  : {n_aug:,}')
    print(f'  Meta          : {MAX_IMAGENES:,}')
    if total_final == MAX_IMAGENES:
        print('  Estado        : ✅ EXACTO — meta lograda')
    elif total_final > MAX_IMAGENES:
        print(f'  Estado        : ⚠️ {total_final - MAX_IMAGENES} de más')
    else:
        print('  Estado        : ⚠️ INCOMPLETO (ni augmentación alcanzó)')
    print(f'  Destino       : {DESTINO_CLASE}')

## CELDA 7 — Verificación final

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

imagenes = sorted(DESTINO_CLASE.glob(f'{CLASE}_*.jpg'))
print(f'Total imágenes en «{CLASE}» : {len(imagenes):,}')

if imagenes:
    sample = random.sample(imagenes, min(100, len(imagenes)))
    corruptas = 0
    for ruta in sample:
        try:
            with Image.open(ruta) as img:
                assert img.size == IMG_SIZE, f'tamaño: {img.size}'
                assert img.mode == 'RGB',    f'modo: {img.mode}'
        except Exception:
            corruptas += 1
    print(f'Verificación ({len(sample)} muestras): '
          f'{len(sample) - corruptas}/{len(sample)} válidas')
    print('✅ Todas 224×224 RGB' if not corruptas else f'⚠️ {corruptas} con problemas')

    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    fig.suptitle(f'Muestra — {CLASE} ({len(imagenes):,} imágenes)', fontsize=13)
    for ax, ruta in zip(axes.flat, random.sample(imagenes, min(8, len(imagenes)))):
        try:
            ax.imshow(mpimg.imread(ruta))
            ax.set_title(ruta.name, fontsize=7)
        except Exception:
            ax.text(0.5, 0.5, 'Error', ha='center', va='center')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print(f'\n🎯 Dataset «{CLASE}» listo en: {DESTINO_CLASE}')

## CELDA 8 — Limpieza del I/O de Colab (corre al terminar)

In [ ]:
# El resultado ya está en Drive; /content/db_raw es efímero y se puede borrar.
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR, ignore_errors=True)
    print(f'✅ Limpiado: {RAW_DIR}')
else:
    print('ℹ️  No hay temporales que limpiar')

print('\n📊 Espacio en disco actual:')
!df -h /content | tail -1